# Omnipose Segmentation QC

This notebook lets you manually QC omnipose segmentations one FOV at a time, using the old cellpose results as a visual reference to guide filtering.

---

### What you see in napari
| Layer | Color | What it is |
|---|---|---|
| phase | gray | phase image — background context |
| cellpose kept | **green** | cells that survived all cellpose QC steps (`seg.npy`) |
| cellpose removed | **red** | cells manually painted out in the old cellpose QC (`seg.manual_qc.npy`) |
| omnipose | **magenta** | all cells found by omnipose in this FOV |
| long cells | **cyan** | cells flagged as too long by the length filter (if available) |
| omnipose labels | colored | full labeled mask — toggle on if you need to identify a cell by ID |
| **bad_regions** | orange | **this is the brush layer — paint over cells you want to remove** |

Green + red together = the full manually-reviewed cellpose population. Toggle one off to focus on just the kept or just the removed cells.

---

### Workflow
1. Edit **Cell 1** (settings) — set the experiment folder and any options.
2. Run **Cell 2** (discovery) — prints which FOVs exist and their QC status.
3. Run **Cell 3** (QC loop) — napari opens for each pending FOV. Paint bad regions, then close the window. The next FOV opens automatically.

### Output files saved per FOV (inside `omnipose_seg/{fov_name}/`)
- **`{fov_name}.seg.npy`** — the filtered labeled mask (cells that passed QC)
- **`{fov_name}.seg.manual_qc.npy`** — the cells you removed (for audit trail)

**Kernel:** `conda activate omnipose`

---
## Cell 1 — Settings

Edit these before running anything else.

- **`EXP_DIR`** — the experiment folder to work on. Just change the last part (e.g. swap `zp_ecoli_M902_p2f_020625` for another experiment name). The four sub-directories (`phase-proj`, `old_seg`, `omnipose_seg`, `long_cell_filter_by_length`) are derived automatically.
- **`HYB`** — the hyb number that appears in the file names (almost always `1`).
- **`FOV_LIST`** — controls which FOVs to process:
  - `None` → all FOVs found in `omnipose_seg`
  - `1` → single FOV
  - `[1, 5, 12]` → explicit list
  - `"1-10"` → range, expands to FOVs 1, 2, 3 … 10
- **`IS_OVERWRITE`** — `False` skips FOVs that already have a `seg.manual_qc.npy` file. Set to `True` to re-open and redo a FOV.
- **`BRUSH_SIZE`** — napari brush radius in pixels. Increase if you want to paint larger areas at once.
- **`SHOW_LONG_CELLS`** — whether to add the cyan long-cell layer (ignored silently if the file doesn't exist for a given FOV).

In [15]:
from pathlib import Path

# ── the only things you need to change ───────────────────────────────────────
_base    = Path.home() / "Box/Zohar_Persky/projects/p2f-revisions/morph-omnipose"
EXP_DIR  = _base / "zp_ecoli_M902_p2f_020625"   # <-- change experiment here

HYB             = 1     # hyb number in file names  (fov_X_hyb_1)
FOV_LIST        = 1  # None = all FOVs in omnipose_seg; or e.g. [1, 2, 5]
IS_OVERWRITE    = True # False = skip already-QC'd FOVs
BRUSH_SIZE      = 10    # napari brush radius in pixels
SHOW_LONG_CELLS = False  # show cyan long-cell flags when the file exists

# ── derived paths (no need to edit) ─────────────────────────────────────────
PHASE_DIR     = EXP_DIR / "phase-proj"                # phase images (.phase.tif)
CELLPOSE_DIR  = EXP_DIR / "old_seg"                   # old cellpose results (reference only)
OMNIPOSE_DIR  = EXP_DIR / "omnipose_seg"              # omnipose results  ← files are written here
LONG_CELL_DIR = EXP_DIR / "long_cell_filter_by_length"

print(f"Experiment : {EXP_DIR.name}")
print(f"omnipose_seg exists: {OMNIPOSE_DIR.exists()}")

Experiment : zp_ecoli_M902_p2f_020625
omnipose_seg exists: True


---
## Cell 2 — Discover FOVs

Scans `omnipose_seg` and builds the list of FOVs to process.

- If `FOV_LIST` is `None`, it finds every sub-folder named `fov_N_hyb_{HYB}` and sorts them numerically.
- If `FOV_LIST` is a list of numbers, it constructs the folder names from those numbers.

For each FOV it prints:
- **`[DONE]`** if `seg.manual_qc.npy` already exists (will be skipped unless `IS_OVERWRITE=True`)
- **`[pending]`** if not yet QC'd
- **`(no cellpose ref)`** if there is no matching folder in `old_seg` (the green reference layer will be absent)
- **`(long cells)`** if long-cell flags are available for that FOV

In [ ]:
import re

# parse FOV_LIST into a flat list of ints, supporting:
#   None        → all FOVs found in omnipose_seg
#   1           → single FOV
#   [1, 2, 5]   → explicit list
#   "1-10"      → range, expands to [1, 2, 3, ..., 10]
if FOV_LIST is None:
    fov_dirs = sorted(
        [d for d in OMNIPOSE_DIR.iterdir()
         if d.is_dir() and re.match(rf'fov_\d+_hyb_{HYB}$', d.name)],
        key=lambda p: int(re.search(r'fov_(\d+)', p.name).group(1))
    )
    FOV_NAMES = [d.name for d in fov_dirs]
else:
    if isinstance(FOV_LIST, str) and '-' in FOV_LIST:
        # range string like "1-10" → [1, 2, ..., 10]
        start, end = FOV_LIST.split('-')
        fov_ints = list(range(int(start), int(end) + 1))
    elif isinstance(FOV_LIST, int):
        fov_ints = [FOV_LIST]
    else:
        fov_ints = list(FOV_LIST)
    FOV_NAMES = [f"fov_{fov}_hyb_{HYB}" for fov in fov_ints]

print(f"Experiment : {EXP_DIR.name}")
print(f"Found {len(FOV_NAMES)} FOV(s):\n")
for name in FOV_NAMES:
    qc_done  = (OMNIPOSE_DIR / name / f"{name}.seg.manual_qc.npy").exists()
    cp_avail = (CELLPOSE_DIR / name / f"{name}.seg.npy").exists()
    lc_avail = (LONG_CELL_DIR / name / f"{name}.seg.long_cells.npy").exists()
    flags = []
    if not cp_avail: flags.append("no cellpose ref")
    if lc_avail:     flags.append("long cells")
    flag_str = f"  ({', '.join(flags)})" if flags else ""
    print(f"  {name}  [{'DONE' if qc_done else 'pending'}]{flag_str}")

---
## Cell 3 — QC loop

Runs through every FOV in `FOV_NAMES` and opens napari for each one.

**What happens for each FOV:**
1. Loads the phase image, the current omnipose labeled mask, the old cellpose binary mask, and the long-cell flags (if available).
2. Opens napari. The `bad_regions` layer is already selected and in paint mode — just start painting.
3. Paint over any cells you want to remove. You don't need to be exact — any omnipose cell whose pixels overlap with your brush strokes will be removed.
4. **Close the napari window** when you're done with this FOV. The code then:
   - Identifies which omnipose cell IDs are under your brush strokes
   - Zeros those cells out of the labeled mask → saves as `seg.npy`
   - Saves the removed cells separately → `seg.manual_qc.npy`
5. Automatically opens the next FOV.

**Tips:**
- If you made no brush strokes (nothing to remove), just close — nothing will be deleted.
- The `omnipose labels` layer is hidden by default. Toggle it on if you want to see each cell's numeric ID.
- To go back and redo a FOV later: set `IS_OVERWRITE=True` and `FOV_LIST=[N]`, then re-run Cell 3. The new removed cells are merged with any previously removed ones.

In [17]:
import numpy as np
import napari
import tifffile

# all known cellpose drop-file suffixes — any that exist get merged into one layer
CELLPOSE_DROP_SUFFIXES = [
    "seg.fluor_dropped.npy",
    "seg.manual_qc.npy",
    "seg.prop_dropped_cells.npy",
    "seg.demult_dropped.npy",
]


def run_qc(fov_name):
    omni_dir = OMNIPOSE_DIR / fov_name
    qc_path  = omni_dir / f"{fov_name}.seg.manual_qc.npy"

    if not IS_OVERWRITE and qc_path.exists():
        print(f"  [{fov_name}] already QC'd — skipping (set IS_OVERWRITE=True to redo)")
        return

    # ── load phase image ─────────────────────────────────────────────────────
    phase_path = PHASE_DIR / f"{fov_name}.phase.tif"
    if not phase_path.exists():
        print(f"  [{fov_name}] WARNING: phase image not found — skipping")
        return
    phase = tifffile.imread(phase_path)

    # ── load current omnipose labeled mask ───────────────────────────────────
    omni_seg_path = omni_dir / f"{fov_name}.seg.npy"
    if not omni_seg_path.exists():
        print(f"  [{fov_name}] WARNING: omnipose seg.npy not found — skipping")
        return
    omni_seg = np.load(omni_seg_path)

    # ── load old cellpose kept mask (green) ──────────────────────────────────
    cp_kept_path = CELLPOSE_DIR / fov_name / f"{fov_name}.seg.npy"
    cp_kept_bin  = (np.load(cp_kept_path) > 0).astype(np.uint8) \
                   if cp_kept_path.exists() else None

    # ── merge all cellpose drop files into one "dropped" layer (red) ─────────
    # looks for every known drop suffix and ORs them together
    cp_dropped_bin = None
    found_drop_files = []
    for suffix in CELLPOSE_DROP_SUFFIXES:
        drop_path = CELLPOSE_DIR / fov_name / f"{fov_name}.{suffix}"
        if drop_path.exists():
            arr = (np.load(drop_path) > 0).astype(np.uint8)
            cp_dropped_bin = arr if cp_dropped_bin is None else np.clip(cp_dropped_bin + arr, 0, 1)
            found_drop_files.append(suffix.replace("seg.", "").replace(".npy", ""))

    # ── load long-cell flags ──────────────────────────────────────────────────
    lc_path = LONG_CELL_DIR / fov_name / f"{fov_name}.seg.long_cells.npy"
    lc_bin  = (np.load(lc_path) > 0).astype(np.uint8) \
              if (SHOW_LONG_CELLS and lc_path.exists()) else None

    print(f"  [{fov_name}]  omnipose: {int(omni_seg.max())} cells", end="")
    if cp_kept_bin is not None:    print("  | cellpose kept: yes", end="")
    if cp_dropped_bin is not None: print(f"  | dropped ({', '.join(found_drop_files)})", end="")
    if lc_bin is not None:         print("  | long-cell flags: yes", end="")
    print()

    # ── build the napari viewer ───────────────────────────────────────────────
    viewer = napari.Viewer(title=f"Omnipose QC — {fov_name}")

    # gray phase image — background
    viewer.add_image(
        phase, name="phase",
        colormap="gray", blending="translucent",
        contrast_limits=[int(phase.min()), int(phase.max())]
    )

    # green = cellpose cells that passed all QC steps
    if cp_kept_bin is not None:
        viewer.add_image(
            cp_kept_bin, name="cellpose kept (green)",
            colormap="green", blending="additive", opacity=0.5,
            contrast_limits=[0, 1]
        )

    # red = all cellpose dropped cells merged (fluor_dropped + manual_qc + any others)
    if cp_dropped_bin is not None:
        viewer.add_image(
            cp_dropped_bin, name="cellpose dropped (red)",
            colormap="red", blending="additive", opacity=0.6,
            contrast_limits=[0, 1]
        )

    # magenta = all omnipose cells
    viewer.add_image(
        (omni_seg > 0).astype(np.uint8), name="omnipose (magenta)",
        colormap="magenta", blending="additive", opacity=0.5,
        contrast_limits=[0, 1]
    )

    # cyan = long-cell flags
    if lc_bin is not None:
        viewer.add_image(
            lc_bin, name="long cells (cyan)",
            colormap="cyan", blending="additive", opacity=0.7,
            contrast_limits=[0, 1]
        )

    # full labeled omnipose mask — hidden by default, toggle on to read cell IDs
    viewer.add_labels(
        omni_seg, name="omnipose labels (toggle for cell IDs)",
        visible=False
    )

    # brush layer — paint here to mark cells for removal
    bad_layer = viewer.add_labels(
        np.zeros(phase.shape, dtype=int),
        name="bad_regions (paint here)"
    )
    bad_layer.brush_size = BRUSH_SIZE
    bad_layer.mode = "paint"

    print(f"  [{fov_name}] viewer open — paint bad regions, then CLOSE the window")
    viewer.show(block=True)

    # ── process brush annotations ─────────────────────────────────────────────
    cells_to_remove = np.unique(omni_seg[bad_layer.data > 0])
    cells_to_remove = cells_to_remove[cells_to_remove != 0]

    filtered_mask = omni_seg.copy()
    filtered_mask[np.isin(filtered_mask, cells_to_remove)] = 0

    manual_mask = omni_seg.copy()
    manual_mask[~np.isin(manual_mask, cells_to_remove)] = 0

    # if re-doing QC, merge with previously removed cells
    if IS_OVERWRITE and qc_path.exists():
        try:
            prev_manual = np.load(qc_path)
            manual_mask = (manual_mask + prev_manual).astype(int)
        except Exception:
            pass

    # ── save ─────────────────────────────────────────────────────────────────
    np.save(omni_dir / f"{fov_name}.seg.npy",           filtered_mask)
    np.save(omni_dir / f"{fov_name}.seg.manual_qc.npy", manual_mask)

    n_removed = len(cells_to_remove)
    n_left    = len(np.unique(filtered_mask[filtered_mask != 0]))
    print(f"  [{fov_name}] removed {n_removed} cells  |  {n_left} remaining  — saved\n")


# ── loop over all FOVs ────────────────────────────────────────────────────────
for fov_name in FOV_NAMES:
    print(f"\nProcessing: {fov_name}")
    run_qc(fov_name)

print("All FOVs processed.")


Processing: fov_1_hyb_1
  [fov_1_hyb_1]  omnipose: 11796 cells  | cellpose kept: yes  | dropped (fluor_dropped, manual_qc)
  [fov_1_hyb_1] viewer open — paint bad regions, then CLOSE the window
  [fov_1_hyb_1] removed 16 cells  |  11594 remaining  — saved

All FOVs processed.
